In [61]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Text Editor Tool class
class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir
        self.backup_dir = backup_dir
       
    def edit_content(self, new_content):
        self.content = new_content

    def get_content(self):
        return self.content

In [87]:
# Helper functions
import json

from anthropic.types import Message
def add_user_message(messages, message):
    user_message = {"role": "user", "content": message.content if isinstance(message, Message) else message}
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message


def get_text_from_message(message):
    return "".join([part.text for part in message.content if isinstance(part, str)])

def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])
        add_assistant_message(messages, response)
        print(f"Assistant: {get_text_from_message(response)}")
        #if response is not for tool use
        if response.stop_reason != "tool_use":
            break

        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)

    return messages

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)

def run_tools(message):
    tool_results = []
    for part in message.content:
        if part.type == "tool_use":
            tool_id = part.id
            name = part.name
            tool_input = part.input
            # Call the tool function based on the tool_id
            try:
                result = run_tool(name, tool_input)
                tool_results.append({"type": "tool_result", "tool_use_id": tool_id, "content": json.dumps(result), "is_error": False})
            except Exception as e:
                tool_results.append({"type": "tool_result", "tool_use_id": tool_id, "content": f"Error occurred while calling {name}: {str(e)}", "is_error": True})
    print(f"Tool results: {tool_results}")
    return tool_results


In [86]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [83]:
from datetime import datetime

from anthropic.types import ToolParam

def get_current_datetime(datetime_format="%Y-%m-%d %H:%M:%S"):
    if not datetime_format:
        raise ValueError("datetime_format cannot be empty or None")
    return datetime.now().strftime(datetime_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format string. This tool retrieves the current system date and time, then formats it based on the provided datetime_format parameter. The format string should follow Python's strftime directives, allowing for a wide range of date and time representations. For example, '%Y-%m-%d %H:%M:%S' will return the date and time in the format '2025-04-03 14:30:00', while '%A, %B %d, %Y' will return 'Thursday, April 03, 2025'. Users can customize the output format to suit their needs, whether for logging, display, or further processing. The tool ensures that the returned datetime string is accurate to the current system time at the moment of invocation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_format": {
                "type": "string",
                "description": "The format string for the output datetime, following Python's strftime directives. This allows users to specify how they want the current date and time to be represented in the returned string. For example, '%Y-%m-%d %H:%M:%S' for a standard timestamp, or '%A, %B %d, %Y' for a more human-readable format. If not provided, the default format will be used.",
            },
        },
        "required": [],
    },
})

In [89]:
messages = []
add_user_message(messages, "set a reminder for my docters appointment which is 143 days after 3rd of March 2034 at 10:30 AM")
run_conversation(messages)

Assistant: 
Tool results: [{'type': 'tool_result', 'tool_use_id': 'toolu_011hpFPwHXBrfMbTW2EF8FkT', 'content': '"Monday, July 24, 2034 10:30:00 AM"', 'is_error': False}]
Assistant: 
----
Setting the following reminder for 2034-07-24T10:30:00:
Doctor's appointment
----
Tool results: [{'type': 'tool_result', 'tool_use_id': 'toolu_01D6DaRSyuYcXpNkY2ctLu5d', 'content': 'null', 'is_error': False}]
Assistant: 


[{'role': 'user',
  'content': 'set a reminder for my docters appointment which is 143 days after 3rd of March 2034 at 10:30 AM'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. First, let me calculate the date that is 143 days after March 3rd, 2034.", type='text'),
   ToolUseBlock(id='toolu_011hpFPwHXBrfMbTW2EF8FkT', caller=DirectCaller(type='direct'), input={'datetime_str': '2034-03-03 10:30:00', 'input_format': '%Y-%m-%d %H:%M:%S', 'duration': 143, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_011hpFPwHXBrfMbTW2EF8FkT',
    'content': '"Monday, July 24, 2034 10:30:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Now let me set the reminder for that date and time:', type='text'),
   ToolUseBlock(id='toolu_01D6DaRSyuYcXpNkY2ctLu5d', caller=Direct

In [76]:
messages = []
add_user_message(messages, "What is the current time formatted as 'YYYY-MM-DD HH:MM:SS'?")

chat_response = chat(messages, tools=[get_current_datetime_schema])
add_assistant_message(messages, chat_response.content)
tool_result = get_current_datetime(**chat_response.content[0].input)
tool_id = chat_response.content[0].id
print(f"Tool Result: {tool_result}, Tool ID: {tool_id}")

Tool Result: 2026-08-10 20:51:35, Tool ID: toolu_01GD9LZz9Xgb8aKMtQDLk4nC


In [65]:
add_user_message(
    messages,
    text=[{"type": "tool_result", "tool_use_id": tool_id, "content": tool_result}],
)
messages

[{'role': 'user',
  'content': "What is the current time formatted as 'YYYY-MM-DD HH:MM:SS'?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_012pmyiEp556F1M2sNGMb7qL', caller=DirectCaller(type='direct'), input={'datetime_format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_012pmyiEp556F1M2sNGMb7qL',
    'content': '2026-08-10 06:40:42'}]}]

In [66]:
response = chat(messages, tools=[get_current_datetime_schema])
response

Message(id='msg_011Cdspe5Po8AdPdfeg7gXKt', container=None, content=[TextBlock(citations=None, text='The current time is **2026-08-10 06:40:42** (August 10, 2026 at 6:40:42 AM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=929, output_tokens=40, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [68]:
response.content[0].text

'The current time is **2026-08-10 06:40:42** (August 10, 2026 at 6:40:42 AM).'